# 04 - Linear models

Predict each participant's *true* generative decline rate (`data/simulated/true_slopes.csv`)
from demographics plus cold-start features - slope/variability/curvature computed
from only their first 2 visits, simulating a brand-new user. Compares a naive
population-mean baseline against Ridge regression, using GroupKFold so no
participant appears in both train and test.

In [1]:
import sys
sys.path.append("..")

import os
import pandas as pd

from src.preprocessing import load_data, truncate_to_early_visits, validate_longitudinal
from src.longitudinal.slope_extraction import extract_slopes
from src.longitudinal.trends import build_trend_features
from src.models.models import build_feature_matrix, get_naive_baseline, build_ridge_model
from src.evaluation.evaluation import evaluate_cross_participant


In [2]:
DOMAINS = ("memory", "attention", "language")

sim_df = validate_longitudinal(load_data("../data/simulated/longitudinal_simulated.csv"))
true_slopes_df = pd.read_csv("../data/simulated/true_slopes.csv", dtype={"participant_id": str})

early_df = truncate_to_early_visits(sim_df, n_visits=2)
early_slope_table = extract_slopes(early_df, domains=DOMAINS)
early_trend_table = build_trend_features(early_df, domains=DOMAINS)

early_slope_table.shape, early_trend_table.shape


c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 99.541350
  warnings.warn(msg, ConvergenceWarning)


c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2206: ConvergenceWarning: MixedLM optimization failed, trying a different optimizer may help.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2218: ConvergenceWarning: Gradient optimization failed, |grad| = 130.438346
  warnings.warn(msg, ConvergenceWarning)
c:\Users\yusef\Downloads\Cognitive-Decline-Prediction-main\.venv\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive defi

((250, 37), (250, 24))

In [3]:
models = {"baseline_mean": get_naive_baseline(), "ridge": build_ridge_model()}

linear_results = []
for domain in DOMAINS:
    X, y, groups = build_feature_matrix(early_slope_table, early_trend_table, true_slopes_df, domain=domain)
    result = evaluate_cross_participant(X, y, groups, models)
    result["domain"] = domain
    linear_results.append(result)

linear_results_df = pd.concat(linear_results, ignore_index=True)
linear_results_df


,model,mode,mae,rmse,r2,n,domain
0,baseline_mean,cross_participant,0.060712,0.075489,-0.004900,250,memory
1,ridge,cross_participant,0.054412,0.066810,0.212904,250,memory
2,baseline_mean,cross_participant,0.041327,0.051553,-0.001703,250,attention
3,ridge,cross_participant,0.041505,0.051606,-0.003762,250,attention
4,baseline_mean,cross_participant,0.035957,0.043218,-0.004245,250,language
5,ridge,cross_participant,0.034383,0.042255,0.040024,250,language


In [4]:
os.makedirs("../data/processed", exist_ok=True)
linear_results_df.to_csv("../data/processed/linear_model_results.csv", index=False)
